_Считаем данные._

In [1]:
import pandas as pd

df = pd.read_csv('train.csv')
print(df.shape)
df.head()

(30471, 292)


,id,timestamp,full_sq,life_sq,floor,max_floor,material,build_year,num_room,kitch_sq,...,cafe_count_5000_price_2500,cafe_count_5000_price_4000,cafe_count_5000_price_high,big_church_count_5000,church_count_5000,mosque_count_5000,leisure_count_5000,sport_count_5000,market_count_5000,price_doc
0,1,2011-08-20,43,27.0,4.0,NaN,NaN,NaN,NaN,NaN,...,9,4,0,13,22,1,0,52,4,5850000
1,2,2011-08-23,34,19.0,3.0,NaN,NaN,NaN,NaN,NaN,...,15,3,0,15,29,1,10,66,14,6000000
2,3,2011-08-27,43,29.0,2.0,NaN,NaN,NaN,NaN,NaN,...,10,3,0,11,27,0,4,67,10,5700000
3,4,2011-09-01,89,50.0,9.0,NaN,NaN,NaN,NaN,NaN,...,11,2,1,4,4,0,0,26,3,13100000
4,5,2011-09-05,77,77.0,4.0,NaN,NaN,NaN,NaN,NaN,...,319,108,17,135,236,2,91,195,14,16331452


_Будем использовать метрику RMSLE. Минимумы MSLE и RMSLE достигаются в одних и тех же точках. Легче для оптимизации использовать MSLE, а уже после брать от неё корень._

### Что такое MSLE?

**MSLE** — _измеряет ошибку предсказаний в логарифмическом масштабе._

$$
\boxed{
MSLE =
\frac{1}{n}
\sum_{i=1}^{n}
\left[
\log(1+y_i)-\log(1+\hat{y}_i)
\right]^2
}
$$


_Сильнее ориентируется на **относительную ошибку**._

_Чтобы работать с ней достаточно просто логарифмировать таргет._

$$
y'=\log(1+y)
$$

_и обучать модель с обычным **MSE** на \(y'\). Это эквивалентно оптимизации MSLE._

In [2]:
import numpy as np
df = df.assign(log_price_doc=np.log1p(df['price_doc']))

log_targ = np.log1p(df['price_doc'])
df['log_price_doc'] = log_targ
df = df.drop('price_doc', axis=1)

C:\Users\user\AppData\Local\Temp\ipykernel_7892\2318920654.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = df.assign(log_price_doc=np.log1p(df['price_doc']))


In [3]:
df.head()

,id,timestamp,full_sq,life_sq,floor,max_floor,material,build_year,num_room,kitch_sq,...,cafe_count_5000_price_2500,cafe_count_5000_price_4000,cafe_count_5000_price_high,big_church_count_5000,church_count_5000,mosque_count_5000,leisure_count_5000,sport_count_5000,market_count_5000,log_price_doc
0,1,2011-08-20,43,27.0,4.0,NaN,NaN,NaN,NaN,NaN,...,9,4,0,13,22,1,0,52,4,15.581952
1,2,2011-08-23,34,19.0,3.0,NaN,NaN,NaN,NaN,NaN,...,15,3,0,15,29,1,10,66,14,15.607270
2,3,2011-08-27,43,29.0,2.0,NaN,NaN,NaN,NaN,NaN,...,10,3,0,11,27,0,4,67,10,15.555977
3,4,2011-09-01,89,50.0,9.0,NaN,NaN,NaN,NaN,NaN,...,11,2,1,4,4,0,0,26,3,16.388123
4,5,2011-09-05,77,77.0,4.0,NaN,NaN,NaN,NaN,NaN,...,319,108,17,135,236,2,91,195,14,16.608603


### **Работа с вещественными колонками.**

#### _Отберем эти колонки и заполним в них пропуски._

In [4]:
#получим датафрейм только с числовыми столбцами
#df.select_dtypes(include='number')
#select_dtypes — выбор столбцов по их типу данных
#include= - какие типы оставить
#'number' — это специальная категория для всех числовых типов

#получим название этих столбцов
numeric_columns = df.select_dtypes(include='number').columns

df.describe()

,id,full_sq,life_sq,floor,max_floor,material,build_year,num_room,kitch_sq,state,...,cafe_count_5000_price_2500,cafe_count_5000_price_4000,cafe_count_5000_price_high,big_church_count_5000,church_count_5000,mosque_count_5000,leisure_count_5000,sport_count_5000,market_count_5000,log_price_doc
count,30471.000000,30471.000000,24088.000000,30304.000000,20899.000000,20899.000000,1.686600e+04,20899.000000,20899.000000,16912.000000,...,30471.000000,30471.000000,30471.000000,30471.000000,30471.000000,30471.000000,30471.000000,30471.000000,30471.000000,30471.000000
mean,15237.917397,54.214269,34.403271,7.670803,12.558974,1.827121,3.068057e+03,1.909804,6.399301,2.107025,...,32.058318,10.783860,1.771783,15.045552,30.251518,0.442421,8.648814,52.796593,5.987070,15.609483
std,8796.501536,38.031487,52.285733,5.319989,6.756550,1.481154,1.543878e+05,0.851805,28.265979,0.880148,...,73.465611,28.385679,5.418807,29.118668,47.347938,0.609269,20.580741,46.292660,4.889219,0.604574
min,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000e+00,0.000000,0.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,11.512935
25%,7620.500000,38.000000,20.000000,3.000000,9.000000,1.000000,1.967000e+03,1.000000,1.000000,1.000000,...,2.000000,1.000000,0.000000,2.000000,9.000000,0.000000,0.000000,11.000000,1.000000,15.371548
50%,15238.000000,49.000000,30.000000,6.500000,12.000000,1.000000,1.979000e+03,2.000000,6.000000,2.000000,...,8.000000,2.000000,0.000000,7.000000,16.000000,0.000000,2.000000,48.000000,5.000000,15.651990
75%,22855.500000,63.000000,43.000000,11.000000,17.000000,2.000000,2.005000e+03,2.000000,9.000000,3.000000,...,21.000000,5.000000,1.000000,12.000000,28.000000,1.000000,7.000000,76.000000,10.000000,15.931766
max,30473.000000,5326.000000,7478.000000,77.000000,117.000000,6.000000,2.005201e+07,19.000000,2014.000000,33.000000,...,377.000000,147.000000,30.000000,151.000000,250.000000,2.000000,106.000000,218.000000,21.000000,18.526041


_Заполним все пропуски средним значением._

In [5]:
for col in numeric_columns:
    mean_for_col = df[col].mean()
    df[col] = df[col].fillna(mean_for_col)

_Проверим, что всё получилось._

In [6]:
df[numeric_columns].isna().sum()

id                    0
full_sq               0
life_sq               0
floor                 0
max_floor             0
                     ..
mosque_count_5000     0
leisure_count_5000    0
sport_count_5000      0
market_count_5000     0
log_price_doc         0
Length: 276, dtype: int64

#### _Изучим корреляции вещественных признаков._

In [7]:
df[numeric_columns].corr()

,id,full_sq,life_sq,floor,max_floor,material,build_year,num_room,kitch_sq,state,...,cafe_count_5000_price_2500,cafe_count_5000_price_4000,cafe_count_5000_price_high,big_church_count_5000,church_count_5000,mosque_count_5000,leisure_count_5000,sport_count_5000,market_count_5000,log_price_doc
id,1.000000,0.021447,0.012953,-0.022441,0.004010,0.004848,-0.006488,0.008200,-0.005935,-0.061480,...,0.018635,0.022958,0.023704,0.012813,0.015181,-0.048797,0.020671,-0.011680,-0.025627,0.144050
full_sq,0.021447,1.000000,0.153837,0.089450,0.057390,0.015820,-0.002532,0.334760,0.009640,-0.038544,...,0.039083,0.040059,0.043560,0.026895,0.028579,0.021568,0.030218,0.001580,-0.041254,0.271408
life_sq,0.012953,0.153837,1.000000,0.038603,0.034483,0.010200,-0.002324,0.169211,0.000651,-0.058546,...,0.024240,0.026800,0.029921,0.014980,0.015187,0.008870,0.019099,-0.012124,-0.043462,0.119971
floor,-0.022441,0.089450,0.038603,1.000000,0.373873,-0.007439,0.000855,-0.004654,-0.006957,-0.083337,...,-0.034680,-0.029541,-0.022228,-0.044265,-0.045475,-0.012222,-0.044093,-0.101769,-0.123534,0.117870
max_floor,0.004010,0.057390,0.034483,0.373873,1.000000,0.045915,-0.000215,-0.014220,0.020345,-0.061203,...,-0.033935,-0.028423,-0.024940,-0.040653,-0.040605,-0.048229,-0.040977,-0.083772,-0.094960,0.078692
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
mosque_count_5000,-0.048797,0.021568,0.008870,-0.012222,-0.048229,0.041689,0.014699,0.051980,0.010915,0.068570,...,0.529434,0.492859,0.481481,0.502028,0.550342,1.000000,0.499376,0.518500,0.242078,0.130745
leisure_count_5000,0.020671,0.030218,0.019099,-0.044093,-0.040977,0.037472,-0.000601,0.049396,-0.005270,-0.002915,...,0.986206,0.975462,0.948391,0.969029,0.975259,0.499376,1.000000,0.809169,0.463053,0.134175
sport_count_5000,-0.011680,0.001580,-0.012124,-0.101769,-0.083772,0.082620,0.004257,0.075427,0.014149,0.143328,...,0.824139,0.778770,0.753384,0.825883,0.847771,0.518500,0.809169,1.000000,0.733897,0.223386
market_count_5000,-0.025627,-0.041254,-0.043462,-0.123534,-0.094960,0.063992,0.005333,0.051672,0.022122,0.203451,...,0.432265,0.380154,0.353270,0.482547,0.514344,0.242078,0.463053,0.733897,1.000000,0.162569


_Воспользуемся функциями с сайта stackoverflow. Чтобы из нашей таблицы узнать пары наиболее коррелирующих признаков._

In [8]:
### Секретные функции для фильтрации признаков

#Создаёт список пар, которые не нужно проверять:
#(A, A) — корреляция признака с самим собой
#(A, B) и (B, A) — это одна и та же корреляция
def get_redundant_pairs(df):
    pairs_to_drop = set()
    cols = df.columns
    for i in range(0, df.shape[1]):
        for j in range(0, i+1):
            pairs_to_drop.add((cols[i], cols[j]))
    return pairs_to_drop

#находит n пар числовых признаков с самой высокой абсолютной корреляцией между ними.
def get_top_abs_correlations(df, n=5):
    au_corr = df.corr().abs().unstack()
    labels_to_drop = get_redundant_pairs(df)
    au_corr = au_corr.drop(labels=labels_to_drop).sort_values(ascending=False)
    return au_corr[0:n]

print("Top Absolute Correlations")
print(get_top_abs_correlations(df[numeric_columns], 50))

Top Absolute Correlations
railroad_station_walk_km              railroad_station_walk_min                1.000000
children_preschool                    0_6_all                                  1.000000
public_transport_station_km           public_transport_station_min_walk        1.000000
children_school                       7_14_all                                 1.000000
metro_min_walk                        metro_km_walk                            1.000000
raion_build_count_with_material_info  raion_build_count_with_builddate_info    0.999990
young_all                             0_13_all                                 0.999885
young_female                          0_13_female                              0.999871
young_male                            0_13_male                                0.999861
full_all                              female_f                                 0.999850
                                      male_f                                   0.999788
young_

_Снова функциями с сайта stackoverflow удалим признаки, с корреляцией больше 0.9_

In [9]:
### Сворованный со stackoverflow код
### Удалим колонки, где корреляция оказывается > 0.9

def correlation(dataset, threshold):
    col_corr = set() # Set of all the names of deleted columns
    corr_matrix = dataset.corr()
    for i in range(len(corr_matrix.columns)):
        for j in range(i):
            if (corr_matrix.iloc[i, j] >= threshold) and (corr_matrix.columns[j] not in col_corr):
                colname = corr_matrix.columns[i] # getting the name of column
                col_corr.add(colname)
                if colname in dataset.columns:
                    del dataset[colname] # deleting the column from the dataset

_Получим датафрейм из невещественных признаков.
Потом возьмём датафрейм с вещественными, удалим из него ненужное и соединим._

In [10]:
#невещественные
df_not_num = df.drop(columns=numeric_columns)

#вещественные
df_num = df[numeric_columns]
print('до:', df_num.shape)

#удалим из вещественных сильно коррелирующие
correlation(df_num, 0.9)
print('после:',df_num.shape)

#соединим
df = pd.concat([df_not_num, df_num], axis=1)
df.shape

до: (30471, 276)
после: (30471, 142)


(30471, 158)

_Снова посмотрим на топ 50 наиболее коррелирующих признаков. И увидим, что стало лучше._

In [11]:
print(get_top_abs_correlations(df_num, 50))

metro_min_avto                park_km                              0.898442
trc_count_1000                trc_count_1500                       0.897982
railroad_station_walk_km      railroad_km                          0.896954
ttk_km                        exhibition_km                        0.896583
office_raion                  office_count_5000                    0.894774
ts_km                         exhibition_km                        0.894750
cafe_count_500                leisure_count_1000                   0.893153
office_sqm_2000               office_count_5000                    0.893115
culture_objects_top_25_raion  office_raion                         0.892094
school_km                     swim_pool_km                         0.891354
ttk_km                        thermal_power_plant_km               0.891102
green_part_3000               green_part_5000                      0.889150
cafe_count_500                cafe_count_500_price_4000            0.889100
university_k

#### _Посмотрим есть ли у нас квазиконстантные признаки._
_Такие признаки, у которых дисперсия (среднее квадратов отклонений) меньше, чем заданный нами параметр._

In [12]:
from sklearn.feature_selection import VarianceThreshold

#инструмент
cutter = VarianceThreshold(threshold=0.1)
#создадим фильтр
cutter.fit(df_num)
#узнаем какие признаки останутся
leave_feature = cutter.get_feature_names_out()
#посмотрим на те, которые удалили
delete_feature = df_num.columns[~df_num.columns.isin(leave_feature)]
#посмотрим на удалённые признаки
df_num[delete_feature]

,green_zone_part,indust_part,green_zone_km,cafe_count_500_price_high,mosque_count_500,mosque_count_1000,mosque_count_1500,mosque_count_2000
0,0.189727,0.000070,0.600973,0,0,0,0,0
1,0.372602,0.049637,0.065321,0,0,0,0,0
2,0.112560,0.118537,0.453172,0,0,0,0,0
3,0.194703,0.069753,0.106125,0,0,0,0,0
4,0.015234,0.037316,0.236214,0,0,0,0,0
...,...,...,...,...,...,...,...,...
30466,0.096270,0.300323,0.139814,0,0,0,0,1
30467,0.065444,0.000078,0.271861,0,0,0,0,0
30468,0.496315,0.007122,0.427248,0,0,0,0,0
30469,0.167526,0.093443,0.227547,0,0,0,0,0


_Так, как их не очень много, мы их оставим. Скорее всего с ними справится регуляризация. Сохраним датафрейм, для дальнейшей работы._

In [13]:
df.to_csv('df_after_lesson1.csv', index=False)